### Read and Process Data

In [0]:
# Import necessary PySpark libraries
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# Initialize Spark session
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()

In [0]:
# Reusable transformation functions for customer data processing
# These functions can be imported and reused across notebooks

def convert_date_column(df, column_name, date_format="yyyy-MM-dd"):
    """
    Convert a string column to DateType.
    
    Args:
        df: Input DataFrame
        column_name: Name of the column to convert
        date_format: Date format string (default: yyyy-MM-dd)
    
    Returns:
        DataFrame with converted date column
    """
    return df.withColumn(column_name, to_date(col(column_name), date_format))


def fill_missing_locations(df, default_value="Unknown"):
    """
    Fill missing values in location columns (city, state, country).
    
    Args:
        df: Input DataFrame
        default_value: Value to use for missing data (default: Unknown)
    
    Returns:
        DataFrame with filled location columns
    """
    return df.fillna({
        'city': default_value,
        'state': default_value,
        'country': default_value
    })


def add_date_features(df, date_column):
    """
    Extract year and month from a date column.
    
    Args:
        df: Input DataFrame
        date_column: Name of the date column
    
    Returns:
        DataFrame with additional year and month columns
    """
    return df.withColumn(
        f"{date_column}_year",
        year(col(date_column))
    ).withColumn(
        f"{date_column}_month",
        month(col(date_column))
    )


def add_window_rankings(df, partition_col, order_col, ascending=False):
    """
    Add ranking columns using window functions.
    
    Args:
        df: Input DataFrame
        partition_col: Column to partition by
        order_col: Column to order by
        ascending: Sort order (default: False for descending)
    
    Returns:
        DataFrame with rank, dense_rank, and row_number columns
    """
    order_expr = col(order_col).asc() if ascending else col(order_col).desc()
    window_spec = Window.partitionBy(partition_col).orderBy(order_expr)
    
    return df.withColumn('rank', rank().over(window_spec)) \
             .withColumn('dense_rank', dense_rank().over(window_spec)) \
             .withColumn('row_number', row_number().over(window_spec))


def filter_by_date(df, date_column, start_date):
    """
    Filter DataFrame for records on or after a specific date.
    
    Args:
        df: Input DataFrame
        date_column: Name of the date column to filter on
        start_date: Starting date (string in yyyy-MM-dd format)
    
    Returns:
        Filtered DataFrame
    """
    return df.filter(col(date_column) >= lit(start_date))


def load_csv_data(spark, path, header=True, infer_schema=True):
    """
    Load CSV data with standard options.
    
    Args:
        spark: SparkSession
        path: Path to CSV file
        header: Whether CSV has header row (default: True)
        infer_schema: Whether to infer schema (default: True)
    
    Returns:
        DataFrame loaded from CSV
    """
    return spark.read.csv(path, header=header, inferSchema=infer_schema)


print("Transformation functions loaded successfully")

In [0]:
# Read customers data using reusable function
# Uses parameterized path if widgets are defined, otherwise falls back to default
try:
    customers_path = f'/Volumes/{dbutils.widgets.get("catalog")}/{dbutils.widgets.get("schema")}/{dbutils.widgets.get("volume")}/{dbutils.widgets.get("input_folder")}/customers.csv'
except:
    customers_path = '/Volumes/workspace/default/dataset/raw/customers.csv'

data = load_csv_data(spark, customers_path)
data.show()


In [0]:
# Count total number of customer records (presentation)
data.count()

In [0]:
# Display the schema (presentation)
data.printSchema()

In [0]:
# Apply date conversion transformation
customers_df = convert_date_column(data, "registration_date")

In [0]:
# Apply data cleaning transformation
customers_df = fill_missing_locations(customers_df)


In [0]:
# Apply feature engineering transformation
customers_df = add_date_features(customers_df, "registration_date")
customers_df.show()

In [0]:
# Exploratory Data Analysis: Count unique values in location columns
# This shows the geographic diversity of the customer base
customers_df.select(
    countDistinct("city").alias("unique_cities")
).show()
customers_df.select(
    countDistinct("state").alias("unique_states")
).show()
customers_df.select(
    countDistinct("country").alias("unique_countries")
).show()


In [0]:
# Find top 5 cities with the most customers
customers_df.groupBy('city').count().orderBy(desc('count')).show(5)

In [0]:
# Alternative approach: Top 5 cities by customer count (using col() method)
customers_df.groupBy("city").count().orderBy(col('count').desc()).show(5)

In [0]:
# Find top 5 state-country combinations with most customers
# Useful for understanding geographic distribution patterns
customers_df.groupBy("state", "country").count().orderBy(col('count').desc()).show(5)

In [0]:
# Create a pivot table showing active vs inactive users by state
# Columns will be the unique values from is_active (True/False)
customers_df.groupBy("state").pivot("is_active").count().show()

In [0]:
# Apply window ranking transformation
customers_df = add_window_rankings(customers_df, 'state', 'registration_date')
                

In [0]:
# Display the ranking results to compare different ranking functions
customers_df.select("name", "state", "is_active", "rank", "dense_rank", "row_number").show()

In [0]:
# Apply date filtering transformation
recent_customers = filter_by_date(customers_df, "registration_date", "2025-01-01")
recent_customers.show()

In [0]:
# Count how many customers registered since 2025
recent_customers.count()

In [0]:
# Find the earliest and latest registration dates for each city
# This shows the customer acquisition timeline per location
customers_df.groupBy("city").agg(
    min("registration_date").alias("oldest_customer"), 
    max("registration_date").alias("newest_customer")
).show()

In [0]:
# Save processed DataFrames to Delta format for ACID transactions, time travel, and optimized performance
# Remove existing Parquet directories to enable clean Delta conversion
try:
    output_path = f'/Volumes/{dbutils.widgets.get("catalog")}/{dbutils.widgets.get("schema")}/{dbutils.widgets.get("volume")}'
except:
    output_path = '/Volumes/workspace/default/dataset'

# Clean up existing data directories
dbutils.fs.rm(output_path + "/processed_customers", True)
dbutils.fs.rm(output_path + "/recent_customers", True)

# Write as Delta tables
customers_df.write.format("delta").mode("overwrite").save(output_path + "/processed_customers")
recent_customers.write.format("delta").mode("overwrite").save(output_path + "/recent_customers")


### Join `orders_df` with `customers_df`

In [0]:
# Display first 5 rows of the customers DataFrame
customers_df.display(5)

In [0]:
# Read orders data using reusable function
# Uses parameterized path if widgets are defined, otherwise falls back to default
try:
    orders_path = f'/Volumes/{dbutils.widgets.get("catalog")}/{dbutils.widgets.get("schema")}/{dbutils.widgets.get("volume")}/{dbutils.widgets.get("input_folder")}/orders.csv'
except:
    orders_path = '/Volumes/workspace/default/dataset/raw/orders.csv'

orders_df = load_csv_data(spark, orders_path)
orders_df.show(5)

In [0]:
# Apply feature engineering to orders (presentation logic preserved)
orders_df = orders_df.withColumn('order_month', month(col('order_date')))
orders_df.show(5)

In [0]:
# Join customers and orders DataFrames on customer_id (inner join)
# This combines customer information with their order history
customers_orders_df =  customers_df.join(orders_df, 'customer_id', 'inner')
customers_orders_df.show(5)

In [0]:
# Display customers DataFrame again for verification
customers_df.display(5)

In [0]:
# Calculate total number of orders per customer
# Orders are grouped by customer_id and sorted in descending order
customers_orders_count = customers_orders_df.groupBy('customer_id').count().orderBy(col('count').desc())
customers_orders_count.show(10)


This section contains automated tests to verify data quality and transformation logic.

In [0]:
# Test 1: Validate date conversion from string to DateType
from pyspark.sql.types import DateType

def test_date_conversion():
    """Verify registration_date is properly converted to DateType"""
    assert isinstance(customers_df.schema['registration_date'].dataType, DateType), \
        "registration_date should be DateType"
    
    # Verify no null dates after conversion
    null_dates = customers_df.filter(col('registration_date').isNull()).count()
    assert null_dates == 0, f"Found {null_dates} null registration dates"
    
    print("Test passed: Date conversion is correct")

test_date_conversion()

In [0]:
# Test 2: Validate missing values are filled with 'Unknown'
def test_missing_value_handling():
    """Verify null values in location columns are replaced with 'Unknown'"""
    location_cols = ['city', 'state', 'country']
    
    for col_name in location_cols:
        null_count = customers_df.filter(col(col_name).isNull()).count()
        assert null_count == 0, f"Found {null_count} null values in {col_name}"
    
    # Verify 'Unknown' exists in the data (assuming original data had nulls)
    unknown_count = customers_df.filter(
        (col('city') == 'Unknown') | 
        (col('state') == 'Unknown') | 
        (col('country') == 'Unknown')
    ).count()
    
    print(f"Test passed: Missing values handled correctly ({unknown_count} 'Unknown' entries)")

test_missing_value_handling()

In [0]:
# Test 3: Validate feature engineering - year and month extraction
def test_feature_engineering():
    """Verify registration_year and registration_month are correctly extracted"""
    # Check columns exist
    assert 'registration_date_year' in customers_df.columns, "registration_date_year column missing"
    assert 'registration_date_month' in customers_df.columns, "registration_date_month column missing"
    
    # Verify year range is reasonable (e.g., between 2020 and current year)
    year_range = customers_df.select(
        min('registration_date_year').alias('min_year'),
        max('registration_date_year').alias('max_year')
    ).collect()[0]
    
    assert year_range.min_year >= 2020, f"Year {year_range.min_year} seems too old"
    from datetime import datetime
    current_year = datetime.now().year
    assert year_range.max_year <= current_year, f"Year {year_range.max_year} is in the future"
    
    # Verify month range is 1-12
    month_range = customers_df.select(
        min('registration_date_month').alias('min_month'),
        max('registration_date_month').alias('max_month')
    ).collect()[0]
    
    assert 1 <= month_range.min_month <= 12, "Month out of valid range"
    assert 1 <= month_range.max_month <= 12, "Month out of valid range"
    
    print(f"Test passed: Feature engineering correct (years: {year_range.min_year}-{year_range.max_year})")

test_feature_engineering()

In [0]:
# Test 4: Validate filtering logic for recent customers
def test_recent_customers_filter():
    """Verify recent_customers contains only customers registered on or after 2025-01-01"""
    from datetime import date
    
    # All dates should be >= 2025-01-01
    cutoff_date = date(2025, 1, 1)
    invalid_dates = recent_customers.filter(col('registration_date') < lit(cutoff_date)).count()
    
    assert invalid_dates == 0, f"Found {invalid_dates} records before 2025-01-01"
    
    # Verify we have some recent customers
    total_recent = recent_customers.count()
    assert total_recent > 0, "No recent customers found"
    
    # Verify recent_customers is a subset of customers_df
    total_customers = customers_df.count()
    assert total_recent <= total_customers, "Recent customers exceeds total customers"
    
    print(f"Test passed: Recent customers filter correct ({total_recent} customers since 2025)")

test_recent_customers_filter()

In [0]:
# Test 5: Validate window function ranking logic
def test_window_functions():
    """Verify rank, dense_rank, and row_number are calculated correctly"""
    # Check columns exist
    required_cols = ['rank', 'dense_rank', 'row_number']
    for col_name in required_cols:
        assert col_name in customers_df.columns, f"{col_name} column missing"
    
    # Verify all ranking values are positive
    for ranking_col in required_cols:
        min_value = customers_df.select(min(ranking_col)).collect()[0][0]
        assert min_value >= 1, f"{ranking_col} has invalid value {min_value}"
    
    # Verify row_number is unique within each state partition
    duplicate_row_nums = customers_df.groupBy('state', 'row_number').count().filter(col('count') > 1).count()
    assert duplicate_row_nums == 0, f"Found {duplicate_row_nums} duplicate row_number values within states"
    
    # Verify ranking relationship: rank >= dense_rank
    # and both should be >= 1 and <= row_number for a given partition
    invalid_rankings = customers_df.filter(
        (col('rank') < col('dense_rank')) |
        (col('rank') < 1) |
        (col('dense_rank') < 1) |
        (col('row_number') < 1)
    ).count()
    
    assert invalid_rankings == 0, f"Found {invalid_rankings} records with invalid ranking values"
    
    print("Test passed: Window functions (rank, dense_rank, row_number) are correct")

test_window_functions()

In [0]:
# Test 6: Validate aggregation logic for customer counts by city
def test_aggregation_by_city():
    """Verify groupBy and count aggregations produce correct results"""
    city_counts = customers_df.groupBy('city').count()
    
    # Verify all counts are positive
    negative_counts = city_counts.filter(col('count') <= 0).count()
    assert negative_counts == 0, "Found non-positive counts"
    
    # Verify sum of city counts equals total customers
    total_by_city = city_counts.agg(sum('count')).collect()[0][0]
    total_customers = customers_df.count()
    assert total_by_city == total_customers, \
        f"Sum of city counts ({total_by_city}) doesn't match total ({total_customers})"
    
    # Verify top city has reasonable count
    top_city = city_counts.orderBy(col('count').desc()).first()
    assert top_city['count'] >= 1, "Top city should have at least 1 customer"
    
    print(f"Test passed: City aggregation correct (top city: {top_city['city']} with {top_city['count']} customers)")

test_aggregation_by_city()

In [0]:
# Test 7: Validate join between customers and orders
def test_join_integrity():
    """Verify customers_orders_df join maintains data integrity"""
    # Verify join produced results
    joined_count = customers_orders_df.count()
    assert joined_count > 0, "Join produced no results"
    
    # Verify all customer_ids in joined data exist in both source tables
    joined_customer_ids = customers_orders_df.select('customer_id').distinct()
    
    # All joined customer_ids should exist in customers_df
    customers_with_orders = customers_df.join(
        joined_customer_ids, 
        'customer_id', 
        'inner'
    ).count()
    
    distinct_joined_customers = joined_customer_ids.count()
    assert customers_with_orders == distinct_joined_customers, \
        "Some joined customer_ids don't exist in customers_df"
    
    # Verify orders_df customer_ids exist in the join
    orders_customer_ids = orders_df.select('customer_id').distinct().count()
    assert distinct_joined_customers <= orders_customer_ids, \
        "Join has more customers than orders dataset"
    
    print(f"Test passed: Join integrity validated ({joined_count} joined records)")

test_join_integrity()

In [0]:
# Test 8: Validate orders per customer aggregation
def test_orders_per_customer():
    """Verify customer order count aggregation is accurate"""
    # Verify all counts are positive
    invalid_counts = customers_orders_count.filter(col('count') <= 0).count()
    assert invalid_counts == 0, "Found non-positive order counts"
    
    # Verify max orders per customer is reasonable
    max_orders = customers_orders_count.agg(max('count')).collect()[0][0]
    assert max_orders >= 1, "Max orders should be at least 1"
    assert max_orders < 10000, f"Max orders {max_orders} seems unreasonably high"
    
    # Verify the aggregation count matches distinct customers in joined data
    distinct_customers_in_agg = customers_orders_count.count()
    distinct_customers_in_join = customers_orders_df.select('customer_id').distinct().count()
    
    assert distinct_customers_in_agg == distinct_customers_in_join, \
        f"Customer count mismatch: aggregation has {distinct_customers_in_agg}, join has {distinct_customers_in_join}"
    
    print(f"Test passed: Orders per customer aggregation correct (max: {max_orders} orders)")

test_orders_per_customer()

In [0]:
# Run all tests and display summary
def run_all_tests():
    """Execute all test functions and provide a summary"""
    test_functions = [
        ('Date Conversion', test_date_conversion),
        ('Missing Value Handling', test_missing_value_handling),
        ('Feature Engineering', test_feature_engineering),
        ('Recent Customers Filter', test_recent_customers_filter),
        ('Window Functions', test_window_functions),
        ('City Aggregation', test_aggregation_by_city),
        ('Join Integrity', test_join_integrity),
        ('Orders Per Customer', test_orders_per_customer)
    ]
    
    print("\n" + "="*60)
    print("RUNNING AUTOMATED TEST SUITE")
    print("="*60 + "\n")
    
    passed = 0
    failed = 0
    
    for test_name, test_func in test_functions:
        try:
            test_func()
            passed += 1
        except AssertionError as e:
            print(f"Test failed: {test_name} - {str(e)}")
            failed += 1
        except Exception as e:
            print(f"Test error: {test_name} - {str(e)}")
            failed += 1
    
    print("\n" + "="*60)
    print(f"TEST SUMMARY: {passed} passed, {failed} failed out of {len(test_functions)} tests")
    print("="*60 + "\n")
    
    return passed, failed

# Run the test suite
run_all_tests()

In [0]:
# Calculate total spend per customer by summing all order amounts
# Results are sorted to show highest-spending customers first
customer_total_spend = customers_orders_df.groupBy('customer_id').agg(
    sum('total_amount').alias('total_spend')
).orderBy(col('total_spend').desc())
customer_total_spend.show(10)

# Store reference to top customers (for potential further analysis)
top_customers = customer_total_spend

In [0]:
# Calculate average order value per customer
# This shows the typical spending amount per order for each customer
customer_average_spend = customers_orders_df.groupBy('customer_id').agg(
    avg('total_amount').alias('total_spend')
).orderBy(col('total_spend').desc())
customer_average_spend.show(10)

In [0]:
# Count orders by their status (e.g., completed, pending, cancelled)
# This helps understand order fulfillment distribution
order_by_status_count = customers_orders_df.groupBy('status').count().orderBy(col('count').desc())
order_by_status_count.show(10)

# Note: The line below appears incomplete (top_products analysis)
top_products = customers_orders_df.count

In [0]:
# Analyze order volume by month to identify seasonal trends
# Results are ordered chronologically (month 1-12)
order_by_month = customers_orders_df.groupBy('order_month').count().orderBy(col('order_month').asc())
order_by_month.show(10)

In [0]:
# Rank customers by total spend using dense_rank
# Dense rank assigns consecutive ranks without gaps (useful for top-N analysis)
window_spec = Window.orderBy(col('total_spend').desc())
dense_ranked_customers = customer_total_spend.withColumn('dense_rank', dense_rank().over(window_spec))
dense_ranked_customers.show(5)

In [0]:
# Identify customers with high order frequency but low total spending
# These are customers who order often but spend less per order (potential upsell targets)
customer_spend_vs_order = customers_orders_count.join(
    customer_total_spend, 'customer_id', 'inner'
).orderBy(col('count').desc(), col('total_spend'))
customer_spend_vs_order.show(5)